In [ ]:
import sys
sys.path.append("../src")

import pandas as pd
import numpy as np

from scipy.optimize import minimize
from scipy.special import expit

from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, log_loss, ConfusionMatrixDisplay

from predict_next_matches import (
    build_elo_feature,
    build_goals_per_game_feature,
    build_xgd_last5_feature,
)
from dixon_coles import build_dixon_coles_feature, DC_FEATURES

In [ ]:
matches = pd.read_csv("../data/processed/all_matches.csv")
matches["MatchDateTime"] = pd.to_datetime(matches["MatchDateTime"])
matches = matches.sort_values("MatchDateTime").reset_index(drop=True)

matches["Season"].unique()

In [ ]:
# The three features the deployed model currently uses
build_elo_feature(matches)
build_goals_per_game_feature(matches)
build_xgd_last5_feature(matches)

In [ ]:
# Fits a Dixon-Coles goal model once per gameweek, on everything before it -
# leak-free, but a real model fit each time, so this cell takes ~10-15s
build_dixon_coles_feature(matches)

matches[["HomeTeam", "AwayTeam"] + DC_FEATURES].tail(10)

In [ ]:
original_features = [
    "EloDiff",
    "XGDDiffLast5",
    "GoalsPerGameDiff"
]

dc_features = DC_FEATURES

all_features = original_features + dc_features

seasons = sorted(matches["Season"].unique())
min_season_matches = 200

seasons

In [ ]:
feature_sets = {
    "Original 3": original_features,
    "Dixon-Coles only": dc_features,
    "All 14 together": all_features,
}

In [ ]:
# Same walk-forward methodology as src/backtest.py: for each season, train
# only on the seasons before it, test on that season, pool the results.
# Run every feature set both unscaled and scaled to see whether scaling
# actually matters here.

comparison_results = []
pooled = {}
model_info = {}  # name -> {"type": ..., "features": [...]}, built up as each model is fit below

for name, features in feature_sets.items():

    for scale in [False, True]:

        pooled_true, pooled_pred, pooled_proba, labels = [], [], [], None
        pooled_fixtures = []

        for index, season in enumerate(seasons):

            if index == 0:
                continue

            train = matches[matches["Season"].isin(seasons[:index])]
            test = matches[matches["Season"] == season]

            if len(test) < min_season_matches:
                continue

            X_train, y_train = train[features], train["FTR"]
            X_test, y_test = test[features], test["FTR"]

            logreg = LogisticRegression(max_iter=2000, random_state=42)

            if scale:
                estimator = Pipeline([("scale", StandardScaler()), ("clf", logreg)])
            else:
                estimator = logreg

            model = CalibratedClassifierCV(estimator=estimator, method="sigmoid", cv=5)
            model.fit(X_train, y_train)
            labels = list(model.classes_)

            proba = model.predict_proba(X_test)
            pred = [labels[i] for i in proba.argmax(axis=1)]

            pooled_true.extend(y_test.tolist())
            pooled_pred.extend(pred)
            pooled_proba.extend(proba.tolist())
            pooled_fixtures.append(test[["Season", "Gameweek", "HomeTeam", "AwayTeam", "FTR"]])

        comparison_results.append({
            "Feature set": name,
            "Scaled": scale,
            "Accuracy": accuracy_score(pooled_true, pooled_pred),
            "LogLoss": log_loss(pooled_true, pooled_proba, labels=labels),
            "Draws predicted": sum(p == "D" for p in pooled_pred),
            "Actual draws": sum(t == "D" for t in pooled_true),
            "Matches": len(pooled_true),
        })

        if scale:
            pooled[name] = {
                "true": pooled_true,
                "pred": pooled_pred,
                "proba": pooled_proba,
                "labels": labels,
                "fixtures": pd.concat(pooled_fixtures, ignore_index=True),
            }
            model_info[name] = {
                "type": "Multinomial Logistic Regression (scaled, sigmoid-calibrated)",
                "features": features,
            }

comparison_df = pd.DataFrame(comparison_results)
comparison_df

In [ ]:
# Scaled vs unscaled side by side - does StandardScaler actually change anything?
comparison_df.pivot(index="Feature set", columns="Scaled", values=["Accuracy", "LogLoss"])

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    pooled["Original 3"]["true"],
    pooled["Original 3"]["pred"],
    labels=pooled["Original 3"]["labels"],
)

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    pooled["Dixon-Coles only"]["true"],
    pooled["Dixon-Coles only"]["pred"],
    labels=pooled["Dixon-Coles only"]["labels"],
)

In [ ]:
# Forward selection: start empty, repeatedly add whichever remaining
# candidate improves pooled log loss the most, stop when nothing helps.
# Always scaled, same walk-forward loop as above. Also keeps the match
# identifiers (Season/HomeTeam/AwayTeam) lined up with each prediction so
# any draw prediction can be traced back to the actual fixture.
#
# class_weight and calibration_method default to the same setup used
# everywhere above (no weighting, sigmoid/Platt calibration) so every
# earlier call to evaluate(features) still behaves identically - both
# are here so the class-weighting and isotonic-vs-sigmoid experiments
# further down can reuse this same walk-forward loop.

def evaluate(features, class_weight=None, calibration_method="sigmoid"):

    pooled_true, pooled_pred, pooled_proba, labels = [], [], [], None
    pooled_fixtures = []

    for index, season in enumerate(seasons):

        if index == 0:
            continue

        train = matches[matches["Season"].isin(seasons[:index])]
        test = matches[matches["Season"] == season]

        if len(test) < min_season_matches:
            continue

        logreg = LogisticRegression(
            max_iter=2000, random_state=42, class_weight=class_weight
        )
        estimator = Pipeline([("scale", StandardScaler()), ("clf", logreg)])
        model = CalibratedClassifierCV(estimator=estimator, method=calibration_method, cv=5)

        model.fit(train[features], train["FTR"])
        labels = list(model.classes_)

        proba = model.predict_proba(test[features])
        pred = [labels[i] for i in proba.argmax(axis=1)]

        pooled_true.extend(test["FTR"].tolist())
        pooled_pred.extend(pred)
        pooled_proba.extend(proba.tolist())
        pooled_fixtures.append(test[["Season", "Gameweek", "HomeTeam", "AwayTeam", "FTR"]])

    return {
        "true": pooled_true,
        "pred": pooled_pred,
        "proba": pooled_proba,
        "labels": labels,
        "fixtures": pd.concat(pooled_fixtures, ignore_index=True),
        "accuracy": accuracy_score(pooled_true, pooled_pred),
        "log_loss": log_loss(pooled_true, pooled_proba, labels=labels),
    }

In [ ]:
remaining = list(all_features)
chosen = []
best_loss = float("inf")
selection_history = []

while remaining:

    round_results = []
    for feature in remaining:
        result = evaluate(chosen + [feature])
        round_results.append((feature, result))

    round_results.sort(key=lambda row: row[1]["log_loss"])
    best_feature, best_result = round_results[0]

    selection_history.append({
        "Round": len(selection_history) + 1,
        "Added": best_feature,
        "LogLoss": best_result["log_loss"],
        "Accuracy": best_result["accuracy"],
        "Improved": best_result["log_loss"] < best_loss - 1e-4,
    })

    if best_result["log_loss"] < best_loss - 1e-4:
        best_loss = best_result["log_loss"]
        chosen.append(best_feature)
        remaining.remove(best_feature)
    else:
        break

selection_history_df = pd.DataFrame(selection_history)
selection_history_df

In [ ]:
print("Selected features:", chosen)

best_combo = evaluate(chosen)
pooled["Forward selected"] = best_combo
model_info["Forward selected"] = {
    "type": "Multinomial Logistic Regression (scaled, sigmoid-calibrated)",
    "features": chosen,
}

print("Accuracy:", best_combo["accuracy"])
print("Log loss:", best_combo["log_loss"])

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    pooled["Forward selected"]["true"],
    pooled["Forward selected"]["pred"],
    labels=pooled["Forward selected"]["labels"],
)

In [ ]:
# Draw predictions, side by side, across every feature set tested
draw_summary = pd.DataFrame([
    {
        "Feature set": name,
        "Draws predicted": sum(p == "D" for p in data["pred"]),
        "Matches": len(data["pred"]),
        "Actual draws": sum(t == "D" for t in data["true"]),
    }
    for name, data in pooled.items()
])
draw_summary

In [ ]:
# Every match ANY of the tested combinations actually predicted a draw for

for name, data in pooled.items():

    fixtures = data.get("fixtures")
    if fixtures is None:
        continue

    results = fixtures.copy()
    results["Predicted"] = data["pred"]

    draws = results[results["Predicted"] == "D"]

    print(f"{name}: {len(draws)} draw prediction(s) out of {len(results)} matches")

    if len(draws):
        display(draws)

In [ ]:
# The best combination's full set of predictions, to look through directly
best_predictions = pooled["Forward selected"]["fixtures"].copy()
best_predictions["Predicted"] = pooled["Forward selected"]["pred"]
best_predictions[["HomeProb", "DrawProb", "AwayProb"]] = pd.DataFrame(
    pooled["Forward selected"]["proba"], columns=pooled["Forward selected"]["labels"]
)[["H", "D", "A"]].to_numpy()

best_predictions

In [ ]:
# Experiment 1: class weighting. Push LogisticRegression to take the Draw
# class more seriously and see whether it ever actually starts predicting
# one, and what that costs in accuracy/log loss. Same scaled walk-forward
# evaluate() as everything above - only class_weight changes.

class_weights_to_try = [1, 1.5, 2, 2.5, 3, 4, 5, 7, 10]

class_weight_results = []

for name, features in [("Original 3", original_features), ("Forward selected", chosen)]:

    for w in class_weights_to_try:

        result = evaluate(features, class_weight={"H": 1, "D": w, "A": 1})

        class_weight_results.append({
            "Feature set": name,
            "Draw weight": w,
            "Accuracy": result["accuracy"],
            "LogLoss": result["log_loss"],
            "Draws predicted": sum(p == "D" for p in result["pred"]),
            "Matches": len(result["pred"]),
        })

class_weight_df = pd.DataFrame(class_weight_results)
class_weight_df

In [ ]:
# The first weight (if any) that gets the forward-selected combo to
# predict a draw at all, and which matches those are
weighted_with_draws = class_weight_df[
    (class_weight_df["Feature set"] == "Forward selected")
    & (class_weight_df["Draws predicted"] > 0)
]

if len(weighted_with_draws):

    first_weight = weighted_with_draws.iloc[0]["Draw weight"]
    print(f"First draw weight to produce any draw predictions: {first_weight}")

    weighted_result = evaluate(chosen, class_weight={"H": 1, "D": first_weight, "A": 1})
    weighted_predictions = weighted_result["fixtures"].copy()
    weighted_predictions["Predicted"] = weighted_result["pred"]

    display(weighted_predictions[weighted_predictions["Predicted"] == "D"])

else:
    print(f"No draw weight up to {class_weights_to_try[-1]} produced any draw predictions.")

In [ ]:
# Experiment 2: isotonic calibration instead of sigmoid (Platt scaling).
# Isotonic is more flexible but needs more data to avoid overfitting the
# calibration curve itself - with ~1,900 training rows that's a real risk,
# so this checks whether it actually helps here or just adds noise.

calibration_results = []

for name, features in [("Original 3", original_features), ("Forward selected", chosen)]:

    for method in ["sigmoid", "isotonic"]:

        result = evaluate(features, calibration_method=method)

        calibration_results.append({
            "Feature set": name,
            "Calibration": method,
            "Accuracy": result["accuracy"],
            "LogLoss": result["log_loss"],
            "Draws predicted": sum(p == "D" for p in result["pred"]),
        })

calibration_df = pd.DataFrame(calibration_results)
calibration_df

In [ ]:
# Experiment 3: ordinal logistic regression (proportional-odds / ordered
# logit), instead of the standard one-vs-rest multinomial LogisticRegression
# used everywhere above. Away/Draw/Home is treated as a single ordered scale
# (Away=0, Draw=1, Home=2) with two cutpoints on top of one linear score,
# rather than three independent decision boundaries - a more natural fit
# for "draw = the middle band", in theory.
#
# No off-the-shelf sklearn model does this, so it's fit by hand with
# scipy.optimize: two cutpoints theta1 < theta2 (theta2 reparametrised as
# theta1 + exp(gap) so the ordering is automatic during optimisation) plus
# one coefficient per feature, maximising the ordinal log-likelihood.

ORDINAL_CODE = {"A": 0, "D": 1, "H": 2}
ORDINAL_LABELS = ["A", "D", "H"]


def ordinal_neg_log_likelihood(params, X, y, n_features):

    beta = params[:n_features]
    theta1 = params[n_features]
    theta2 = theta1 + np.exp(params[n_features + 1])

    score = X @ beta
    p_le_away = expit(theta1 - score)
    p_le_draw = expit(theta2 - score)

    eps = 1e-12
    p = np.where(
        y == 0, p_le_away,
        np.where(y == 1, np.clip(p_le_draw - p_le_away, eps, 1), 1 - p_le_draw),
    )
    return -np.log(np.clip(p, eps, 1)).sum()


def fit_ordinal_logit(X, y):

    n_features = X.shape[1]
    x0 = np.zeros(n_features + 2)
    x0[n_features] = -0.5  # initial theta1; theta2 starts one unit above it

    result = minimize(
        ordinal_neg_log_likelihood, x0, args=(X, y, n_features), method="BFGS"
    )

    beta = result.x[:n_features]
    theta1 = result.x[n_features]
    theta2 = theta1 + np.exp(result.x[n_features + 1])

    return beta, theta1, theta2


def predict_ordinal_proba(X, beta, theta1, theta2):

    score = X @ beta
    p_le_away = expit(theta1 - score)
    p_le_draw = expit(theta2 - score)

    return np.column_stack([p_le_away, p_le_draw - p_le_away, 1 - p_le_draw])

In [ ]:
def evaluate_ordinal(features):

    pooled_true, pooled_pred, pooled_proba = [], [], []
    pooled_fixtures = []

    for index, season in enumerate(seasons):

        if index == 0:
            continue

        train = matches[matches["Season"].isin(seasons[:index])]
        test = matches[matches["Season"] == season]

        if len(test) < min_season_matches:
            continue

        scaler = StandardScaler().fit(train[features])
        X_train = scaler.transform(train[features])
        X_test = scaler.transform(test[features])
        y_train = train["FTR"].map(ORDINAL_CODE).to_numpy()

        beta, theta1, theta2 = fit_ordinal_logit(X_train, y_train)
        proba = predict_ordinal_proba(X_test, beta, theta1, theta2)
        pred = [ORDINAL_LABELS[i] for i in proba.argmax(axis=1)]

        pooled_true.extend(test["FTR"].tolist())
        pooled_pred.extend(pred)
        pooled_proba.extend(proba.tolist())
        pooled_fixtures.append(test[["Season", "Gameweek", "HomeTeam", "AwayTeam", "FTR"]])

    return {
        "true": pooled_true,
        "pred": pooled_pred,
        "proba": pooled_proba,
        "labels": ORDINAL_LABELS,
        "fixtures": pd.concat(pooled_fixtures, ignore_index=True),
        "accuracy": accuracy_score(pooled_true, pooled_pred),
        "log_loss": log_loss(pooled_true, pooled_proba, labels=ORDINAL_LABELS),
    }


ordinal_results = []

for name, features in [("Original 3", original_features), ("Forward selected", chosen)]:

    result = evaluate_ordinal(features)
    pooled[f"{name} (ordinal)"] = result
    model_info[f"{name} (ordinal)"] = {
        "type": "Ordinal Logistic Regression (scaled, proportional-odds)",
        "features": features,
    }

    ordinal_results.append({
        "Feature set": name,
        "Accuracy": result["accuracy"],
        "LogLoss": result["log_loss"],
        "Draws predicted": sum(p == "D" for p in result["pred"]),
        "Matches": len(result["pred"]),
    })

ordinal_df = pd.DataFrame(ordinal_results)
ordinal_df

In [ ]:
# Sanity check: is a draw ever even Dixon-Coles' OWN top pick, before any
# logistic regression - ordinal or otherwise - gets involved at all?
# (across the WHOLE dataset, including 2021/22, which is only ever used
# as training data below and never actually backtested)
dc_argmax = matches[["DCHomeProb", "DCDrawProb", "DCAwayProb"]].idxmax(axis=1)
print(dc_argmax.value_counts())

most_drawish = matches.loc[matches["DCDrawProb"].idxmax()]
most_drawish[["Season", "HomeTeam", "AwayTeam", "DCHomeProb", "DCDrawProb", "DCAwayProb"]]

In [ ]:
# Same check, but restricted to the four seasons actually backtested above
# (22-23 through 25-26) - 2021/22 is never scored, only ever trained on,
# and 2026/27 is still in progress and too small to backtest either, so
# neither should count toward "does this ever pick a draw in practice".
tested_seasons = [
    season for season in seasons[1:]
    if len(matches[matches["Season"] == season]) >= min_season_matches
]
in_window = matches[matches["Season"].isin(tested_seasons)]

print(in_window[["DCHomeProb", "DCDrawProb", "DCAwayProb"]].idxmax(axis=1).value_counts())
print()
print("Highest DCDrawProb actually seen in a backtested season:", in_window["DCDrawProb"].max())

closest_call = in_window.loc[in_window["DCDrawProb"].idxmax()]
closest_call[["Season", "Gameweek", "HomeTeam", "AwayTeam", "DCHomeProb", "DCDrawProb", "DCAwayProb"]]

In [ ]:
# Experiment 4: blend Dixon-Coles' own raw probabilities directly with the
# logistic regression's, instead of feeding DC in as features for another
# model to re-learn. alpha_dc = 0 is pure logistic regression, 1 is pure
# Dixon-Coles; no refitting involved, just averaging two already-computed
# probability sets for the exact same backtested matches.

def blend_with_dixon_coles(pooled_result, alpha_dc):

    fixtures = pooled_result["fixtures"]
    labels = pooled_result["labels"]
    dc_column = {"A": "DCAwayProb", "D": "DCDrawProb", "H": "DCHomeProb"}

    key_cols = ["Season", "Gameweek", "HomeTeam", "AwayTeam"]
    dc_lookup = matches.set_index(key_cols)[[dc_column[label] for label in labels]]
    keys = list(zip(*[fixtures[col] for col in key_cols]))
    dc_proba = dc_lookup.loc[keys].to_numpy()

    lr_proba = np.array(pooled_result["proba"])

    blended = alpha_dc * dc_proba + (1 - alpha_dc) * lr_proba
    blended = blended / blended.sum(axis=1, keepdims=True)

    pred = [labels[i] for i in blended.argmax(axis=1)]

    return {
        "true": pooled_result["true"],
        "pred": pred,
        "proba": blended.tolist(),
        "labels": labels,
        "fixtures": fixtures,
        "accuracy": accuracy_score(pooled_result["true"], pred),
        "log_loss": log_loss(pooled_result["true"], blended, labels=labels),
    }


alphas_to_try = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]

blend_results = []

for name in ["Original 3", "Forward selected"]:
    for alpha in alphas_to_try:

        blended = blend_with_dixon_coles(pooled[name], alpha)

        blend_results.append({
            "Feature set": name,
            "DC weight": alpha,
            "Accuracy": blended["accuracy"],
            "LogLoss": blended["log_loss"],
            "Draws predicted": sum(p == "D" for p in blended["pred"]),
        })

blend_df = pd.DataFrame(blend_results)
blend_df

In [ ]:
# Best blend weight by log loss, for each feature set - added into the
# final summary below alongside everything else that's been tried
for name in ["Original 3", "Forward selected"]:

    subset = blend_df[blend_df["Feature set"] == name]
    best_alpha = subset.loc[subset["LogLoss"].idxmin(), "DC weight"]

    blended_name = f"{name} blended (DC weight {best_alpha})"

    pooled[blended_name] = blend_with_dixon_coles(pooled[name], best_alpha)
    model_info[blended_name] = {
        "type": (
            f"{model_info[name]['type']} blended {best_alpha:.0%} with "
            f"raw Dixon-Coles H/D/A output (no refitting)"
        ),
        "features": model_info[name]["features"],
    }

    print(f"{name}: best DC weight = {best_alpha}")

In [ ]:
# Every model tried, with what it actually is (type + features) alongside
# how it did - the full record of this notebook in one table.
final_summary = pd.DataFrame([
    {
        "Model": name,
        "Type": model_info[name]["type"],
        "Features": ", ".join(model_info[name]["features"]),
        "Accuracy": accuracy_score(data["true"], data["pred"]),
        "LogLoss": log_loss(data["true"], data["proba"], labels=data["labels"]),
        "Draws predicted": sum(p == "D" for p in data["pred"]),
        "Matches": len(data["pred"]),
    }
    for name, data in pooled.items()
]).sort_values("LogLoss").reset_index(drop=True)

final_summary